# 10 – Results: Conclusiones

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas` (1 = subempleado por horas · 0 = no subempleado)  
**Objetivo:** Sintetizar los hallazgos del proyecto y sus implicaciones para política laboral.

> **Prerequisito:** Ejecuta todos los notebooks anteriores (01 → 10) para que los artefactos existan.  
> Los resultados se leen desde `data/results/model_comparison.csv` y `data/results/final_metrics.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path('../data/results')

# ── Carga de tabla comparativa real (sin fallback) ─────────────────────────────
comparison_path = RESULTS_DIR / 'model_comparison.csv'
if not comparison_path.exists():
    raise FileNotFoundError(
        f"Archivo no encontrado: {comparison_path}\n"
        "Ejecuta primero: 07_modelling/03_model_comparison.ipynb"
    )

comparison_df = pd.read_csv(comparison_path)
print(f'Modelos cargados: {len(comparison_df)}')

# Renombrar columnas para visualización
col_map = {
    'model_name':  'Modelo',
    'precision_1': 'Precision (clase 1)',
    'recall_1':    'Recall (clase 1)',
    'f1_1':        'F1 (clase 1)',
    'roc_auc':     'ROC-AUC',
    'accuracy':    'Accuracy',
}
display_df = comparison_df.rename(columns=col_map)

# Mostrar ranking por F1 clase 1
print('\nRanking por F1-score clase 1 (subempleado por horas):')
display_df.sort_values('F1 (clase 1)', ascending=False)[
    ['Modelo', 'Accuracy', 'Precision (clase 1)', 'Recall (clase 1)', 'F1 (clase 1)', 'ROC-AUC']
]


## 1. Resumen ejecutivo del proyecto

| Etapa | Descripción |
|---|---|
| **Fuente de datos** | INEI – EPEN 2024 (Encuesta Permanente de Empleo Nacional) |
| **Universo analítico** | 24 054 trabajadores ocupados del Perú (≥ 14 años, residentes habituales) |
| **Problema** | Clasificación binaria: predecir si un trabajador presenta **subempleo por insuficiencia de horas** |
| **Variable objetivo** | `target_subempleo_horas` (construida desde `P209H`, `C333` y `C334`) |
| **Desbalance de clases** | ~75 % clase 0 (No subempleado) / ~25 % clase 1 (Subempleado por horas) |
| **Estrategia de desbalance** | `class_weight='balanced'` en modelos de clasificación; RandomOverSampler y RandomUnderSampler como análisis complementario |
| **Feature Eng.** | Variables derivadas de ingreso, empleo, educación y demografía |
| **Selección de variables** | Información mutua, RF Importance y SelectFromModel (sobre X_train únicamente) |
| **Modelo final** | Logistic Regression (`class_weight='balanced'`) — mayor F1 clase 1 y ROC-AUC |
| **División train/test** | 80 % / 20 % estratificado · random_state=42 (19 243 / 4 811 observaciones) |


## 2. Desempeño del modelo final

In [ ]:
# Cargar métricas finales (generadas por 09_evaluation/metrics.ipynb)
metrics_path = RESULTS_DIR / 'final_metrics.csv'
if not metrics_path.exists():
    raise FileNotFoundError(
        f"Archivo no encontrado: {metrics_path}\n"
        "Ejecuta primero: 09_evaluation/metrics.ipynb"
    )

metrics_df = pd.read_csv(metrics_path)
print('=== Métricas del modelo final – Logistic Regression (class_weight="balanced") ===\n')
print(metrics_df.T.to_string(header=False))


## 3. Comparación visual de modelos

In [ ]:
metrics_cols = ['Precision (clase 1)', 'Recall (clase 1)', 'F1 (clase 1)', 'ROC-AUC']

plot_df = display_df.set_index('Modelo')[metrics_cols].sort_values('F1 (clase 1)', ascending=False)

ax = plot_df.plot(
    kind='bar', figsize=(13, 5), edgecolor='black',
    color=['#2196F3', '#F44336', '#FF9800', '#4CAF50'],
)
plt.title('Comparación de modelos – Métricas clase 1 (subempleado por horas) en conjunto de prueba')
plt.ylabel('Score')
plt.xticks(rotation=25, ha='right', fontsize=9)
plt.legend(loc='upper right', fontsize=9)
plt.ylim(0, 1.05)
plt.axhline(0.25, linestyle='--', color='gray', linewidth=0.8, label='Baseline prevalencia')
plt.tight_layout()
plt.show()


## 4. Hallazgos principales

1. **La Regresión Logística balanceada es el mejor modelo**, con F1 clase 1 ≈ 0.475 y ROC-AUC ≈ 0.697 en el conjunto de prueba, superando al Dummy Classifier (ROC-AUC = 0.50) y a los modelos de árboles sin optimización. El ajuste por desbalance (`class_weight='balanced'`) fue esencial para detectar la clase minoritaria.

2. **Variables más asociadas al subempleo por horas** (según MDI del RF optimizado y coeficientes LR):
   - **Horas trabajadas** (`C318_T`, `whoraT`): quienes trabajan menos horas tienen mayor probabilidad de ser subempleados por horas — relación directa con la definición del target.
   - **Búsqueda activa de otro empleo** (`busca_otro_empleo`): señal fuerte de insatisfacción laboral.
   - **Condición de la jornada** (`C335_1`, `C335_2`): indicadores de jornada incompleta o irregular.
   - **Ingreso por hora** (`ingreso_por_hora`, `ingreso_por_hora_log`): ingresos bajos se asocian al subempleo.
   - **Protección social** (`SEGURO1`, `tiene_sis`, `tiene_pension`): menor protección está correlacionada con mayor subempleo.

3. **El balance entre Precisión y Recall** es el principal trade-off del modelo. La LR balanceada mejora el Recall de la clase 1 a ~0.57, a costa de una menor Precisión (~0.41) y Accuracy global (~0.69). En el contexto de política pública, maximizar el Recall es prioritario para no dejar subempleados sin identificar.

4. **El umbral de clasificación por defecto (0.5) no es necesariamente el óptimo.** Un análisis de umbral (ver notebook `metrics.ipynb`) puede ajustarlo según el objetivo: mayor Recall o mayor Precisión.

5. **Los modelos de árboles** (Decision Tree GridSearchCV: F1=0.459, RF GridSearchCV: F1=0.454) son competitivos pero no superan a la LR balanceada. Sin embargo, ofrecen mayor interpretabilidad mediante importancia de variables.

> **Advertencia:** Los resultados son correlacionales. El modelo identifica patrones, pero no implica causalidad entre las variables y el subempleo por horas.


## 5. Recomendaciones para política pública

- **Priorizar intervenciones para trabajadores con jornadas cortas involuntarias.** El subempleo por horas concentra a personas que trabajan menos de lo que desean: programas de intermediación laboral y acceso a empleo de tiempo completo son la respuesta directa.

- **Fortalecer el acceso a protección social.** La falta de seguro de salud y pensiones (`tiene_sis`, `tiene_pension`) está correlacionada con mayor subempleo. Extender la cobertura puede reducir la precariedad laboral.

- **Atender la insuficiencia de ingresos como factor acompañante.** Los bajos ingresos por hora (`ingreso_por_hora`) son un predictor fuerte. Políticas salariales y de formalización pueden ayudar.

- **Usar el modelo como herramienta de focalización, no de decisión automatizada.** Los resultados deben complementar, no reemplazar, el análisis humano. El modelo tiene un Recall de ~0.57, lo que implica que ~43 % de los subempleados no son detectados correctamente.

- **Actualizar periódicamente el modelo.** La EPEN se realiza trimestralmente. Reentrenar con cada ola mejora la vigencia de las predicciones.
